# Temporal Dynamics of LWS Probability After a Hit (Q2 - Adamo et al. 2013 replication)

Does `P[LWS]` depend on how recently a target was identified? Two lag measures, modeled in parallel since
they encode different accounts:

- `time_since_recent_find` (ms) - the literal Adamo-style attentional-blink lag.
- `fixations_since_last_hit` (count) - a fixation-lag alternative robust to raw-time/fixation-rate
  confounds, natural here since this dataset is eye-tracked rather than RT-based.

Restricted to visits where `num_targets_found_before > 0` (both lags are undefined otherwise). The shape to
look for is a dip-then-recovery (classic attentional blink) vs. a monotonic decay (pure
resource-depletion/satisfaction account) - not just "is the slope significant."

In [ ]:
import os
from itertools import product

from analysis.helpers.r_bridge import (
    setup_rpy2, to_r_dataframe, source_r, get_r_object, gam_metrics, predict_gam, placeholder_level, cached_fit,
)
setup_rpy2()

import pytensor
pytensor.config.cxx = ""
import bambi as bmb
import arviz as az

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from analysis.ssm_and_ab.ssm import load_ssm_funnel

pio.renderers.default = "notebook"      # "notebook" or "browser"

In [ ]:
data, funnel, hits = load_ssm_funnel()
lag_data = funnel.loc[funnel["num_targets_found_before"] > 0].copy()
print(f"{len(lag_data)} visits with a prior hit ({lag_data['is_lws'].mean():.3f} overall LWS rate)")
lag_data[["time_since_recent_find", "fixations_since_last_hit"]].describe()

### Descriptive: binned LWS rate by lag

In [ ]:
def _binned_rate(df, col, n_bins=10):
    bins = pd.qcut(df[col], q=n_bins, duplicates="drop")
    return df.groupby(bins, observed=True).agg(
        x=(col, "mean"), lws_rate=("is_lws", "mean"), n=("is_lws", "size"),
    ).reset_index(drop=True)


time_binned = _binned_rate(lag_data, "time_since_recent_find")
fix_binned = _binned_rate(lag_data, "fixations_since_last_hit")

fig = make_subplots(rows=1, cols=2, subplot_titles=["Time Since Last Hit (ms)", "Fixations Since Last Hit"])
fig.add_trace(go.Scatter(x=time_binned["x"], y=time_binned["lws_rate"], mode="markers+lines",
                          marker=dict(size=time_binned["n"] / time_binned["n"].max() * 20 + 4)), row=1, col=1)
fig.add_trace(go.Scatter(x=fix_binned["x"], y=fix_binned["lws_rate"], mode="markers+lines",
                          marker=dict(size=fix_binned["n"] / fix_binned["n"].max() * 20 + 4)), row=1, col=2)
fig.update_layout(title="Empirical P[LWS] by Lag Since Last Hit (equal-count bins, marker size ~ n)",
                   template="plotly_white", showlegend=False, width=1000, height=400)
fig.show()

### Frequentist GAMs (flat vs. subject/trial-nested, `CODE_REVIEW.md` M10)

In [ ]:
def _fit_ssm_temporal_gam():
    model_data = lag_data[[
        "subject", "trial", "trial_category", "time_since_recent_find", "fixations_since_last_hit", "is_lws",
    ]].copy()
    to_r_dataframe(model_data, "dat", build_trial_uid=True)
    source_r(os.path.join(os.getcwd(), "..", "R", "ssm_temporal_dynamics_gam.R"))

    metrics = {
        name: gam_metrics(get_r_object(name), name)
        for name in ["model_time_flat", "model_time_nested", "model_fix_flat", "model_fix_nested"]
    }

    subjects = model_data["subject"].astype(str).unique().tolist()
    categories = model_data["trial_category"].cat.categories.tolist()
    trial_uid_level = placeholder_level("dat$trial_uid")

    time_grid = pd.DataFrame(
        product(
            np.linspace(model_data["time_since_recent_find"].min(), model_data["time_since_recent_find"].max(), 60),
            categories, subjects,
        ),
        columns=["time_since_recent_find", "trial_category", "subject"],
    ).assign(trial_uid=trial_uid_level)
    time_grid["prob"] = predict_gam(get_r_object("model_time_nested"), time_grid, exclude=["s(trial_uid)"])

    fix_grid = pd.DataFrame(
        product(range(int(model_data["fixations_since_last_hit"].max()) + 1), categories, subjects),
        columns=["fixations_since_last_hit", "trial_category", "subject"],
    ).assign(trial_uid=trial_uid_level)
    fix_grid["prob"] = predict_gam(get_r_object("model_fix_nested"), fix_grid, exclude=["s(trial_uid)"])

    return {"metrics": metrics, "time_grid": time_grid, "fix_grid": fix_grid}


ssm_temporal_gam_result = cached_fit(
    os.path.join(os.getcwd(), "..", "R", "_cache", "ssm_temporal_dynamics_gam.pkl"), _fit_ssm_temporal_gam,
)

pd.DataFrame([
    {"model": k, "aic": v["aic"], "dev_expl": v["dev_expl"], "n": v["n"]}
    for k, v in ssm_temporal_gam_result["metrics"].items()
])

In [ ]:
for name in ["model_time_nested", "model_fix_nested"]:
    print(f"--- {name} ---")
    display(ssm_temporal_gam_result["metrics"][name]["smooth_terms"])

### Predicted P[LWS] curve

In [ ]:
def _ribbon_plot(grid, x_col, title):
    subj_agg = grid.groupby(["subject", x_col])["prob"].mean().reset_index()
    pop = subj_agg.groupby(x_col).agg(mean_prob=("prob", "mean"), sem_prob=("prob", "sem")).reset_index()
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=pd.concat([pop[x_col], pop[x_col][::-1]]),
        y=pd.concat([pop["mean_prob"] + 1.96 * pop["sem_prob"], (pop["mean_prob"] - 1.96 * pop["sem_prob"])[::-1]]),
        fill="toself", fillcolor="rgba(0,0,0,0.15)", line=dict(color="rgba(0,0,0,0)"),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=pop[x_col], y=pop["mean_prob"], mode="lines", line=dict(color="black", width=3), showlegend=False,
    ))
    fig.update_layout(title=title, template="plotly_white", width=600, height=400,
                       yaxis=dict(title="Predicted P[LWS]"))
    return fig


_ribbon_plot(
    ssm_temporal_gam_result["time_grid"], "time_since_recent_find", "GAM-Predicted P[LWS] vs. Time Since Last Hit",
).show()
_ribbon_plot(
    ssm_temporal_gam_result["fix_grid"], "fixations_since_last_hit", "GAM-Predicted P[LWS] vs. Fixations Since Last Hit",
).show()

### Bayesian companion

`bambi`'s formula parser in this environment supports neither `mgcv`-style `s()` smooths nor `bs()`
B-splines (both raise a `formulae`/prior-scaling error here). The Bayesian companion instead uses a
quadratic parametric restriction of the smooth (mean-centered `time_since_recent_find` + its square) - the
same "parametric restriction of a GAM smooth" approach `CODE_REVIEW.md` M12's log-linear robustness check
already uses elsewhere in this repo. A quadratic can represent either a dip-then-recovery (concave) or an
effectively monotonic shape (if the quadratic term is negligible), which is exactly the qualitative
question here.

In [ ]:
bayes_data = lag_data[["subject", "trial", "trial_category", "time_since_recent_find", "is_lws"]].copy()
bayes_data["time_c"] = bayes_data["time_since_recent_find"] - bayes_data["time_since_recent_find"].mean()
bayes_data["time_c2"] = bayes_data["time_c"] ** 2

bayes_cache_path = os.path.join(os.getcwd(), "..", "R", "_cache", "ssm_temporal_bayes_idata.nc")
bayes_model = bmb.Model("is_lws ~ trial_category + time_c + time_c2 + (1|subject/trial)", bayes_data, family="bernoulli")

if os.path.exists(bayes_cache_path):
    bayes_idata = az.from_netcdf(bayes_cache_path)
    print(f"Loaded cached idata from {bayes_cache_path}")
else:
    bayes_idata = bayes_model.fit(
        draws=2000, tune=1000, chains=4, cores=1, target_accept=0.95, random_seed=42, progressbar=False,
    )
    os.makedirs(os.path.dirname(bayes_cache_path), exist_ok=True)
    az.to_netcdf(bayes_idata, bayes_cache_path)
    print(f"Fit complete, saved to {bayes_cache_path}")

az.summary(bayes_idata, var_names=["Intercept", "time_c", "time_c2"])

In [ ]:
posterior_quad = bayes_idata["posterior"]["time_c2"]
print(f"P[quadratic term > 0] = {(posterior_quad > 0).mean().item():.4f}   "
      f"(positive -> convex/monotonic-ish; negative -> concave/dip-then-recovery)")
az.plot_posterior(bayes_idata, var_names=["time_c", "time_c2"], hdi_prob=0.95)